In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, window, count
from src.schema import transaction_schema
from pyspark.sql import Window, functions as F                                                                                                                 
from config import processed_data_path, checkpoint_path        
print("Checkpoint path:", checkpoint_path)                                                
import sys                                                                                                                                                                      
sys.path.append("../..")                                                                     
from src.data_processing import process_and_write_batch     

Checkpoint path: c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\data\checkpoints\transactions_parquet


In [3]:
spark = SparkSession.builder \
        .appName("KafkaIncrementalBatch") \
        .master("local[*]") \
        .config(
            "spark.jars.packages", 
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
        ) \
        .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.2.0


In [ ]:
df_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "transaction-stream")
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 100)
    .load()
)

parsed_stream_df = (
    df_stream
    .select(
        from_json(
            col("value").cast("string"), 
            transaction_schema
        ).alias("data")
    )
    .select("data.*")
)

query = (
    parsed_stream_df.writeStream
    .foreachBatch(process_and_write_batch)
    .option("checkpointLocation", checkpoint_path.as_posix())
    # .trigger(processingTime="10 seconds")
    .start()
)

query.awaitTermination()

Processed and wrote batch 2452


c:\Users\admin\Desktop\data_science_projects\real_time_fraud_transaction_detection\dev_env\Lib\site-packages\pyspark\sql\classic\dataframe.py:178: UserWarning: The return type of DataFrame.toJSON will be changed from RDD to DataFrame in future releases.
  warnings.warn(


Processed and wrote batch 2453
Processed and wrote batch 2454
Processed and wrote batch 2455
Processed and wrote batch 2456
Processed and wrote batch 2457
Processed and wrote batch 2458
Processed and wrote batch 2459
Processed and wrote batch 2460
Processed and wrote batch 2461
Processed and wrote batch 2462
Processed and wrote batch 2463
Processed and wrote batch 2464
Processed and wrote batch 2465
Processed and wrote batch 2466
Processed and wrote batch 2467
Processed and wrote batch 2468
Processed and wrote batch 2469
Processed and wrote batch 2470
Processed and wrote batch 2471
Processed and wrote batch 2472
Processed and wrote batch 2473
Processed and wrote batch 2474
Processed and wrote batch 2475
Processed and wrote batch 2476
Processed and wrote batch 2477
Processed and wrote batch 2478
Processed and wrote batch 2479
Processed and wrote batch 2480
Processed and wrote batch 2481
Processed and wrote batch 2482
Processed and wrote batch 2483
Processed and wrote batch 2484
Processe